# Home Energy Management — Modeling & Evaluation

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import tensorflow as tf

sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from src.utils.helpers import load_config
from src.data import HEMSDataLoader, FeatureEngineer
from src.models import ModelFactory, Trainer
from src.evaluation import TimeSeriesEvaluator

config = load_config()

In [ ]:
loader = HEMSDataLoader(config)
df = loader.load()

fe = FeatureEngineer(config)
df = fe.build_features(df)

train_df, val_df, test_df = loader.temporal_split(df)
print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

In [ ]:
target_cols = list(config['targets'].values())
feature_cols = fe.get_feature_columns(df)
seq_len = config['model']['sequence_length']

def make_sequences(df_subset, seq_len):
    if len(df_subset) < seq_len:
        return np.empty((0, seq_len, len(feature_cols))), {t: np.empty((0, 1)) for t in target_cols}
    X, y = [], {t: [] for t in target_cols}
    values = df_subset[feature_cols].values
    targets = {t: df_subset[t].values for t in target_cols}
    for i in range(len(values) - seq_len - 1):
        X.append(values[i:i+seq_len])
        for t in target_cols:
            y[t].append(targets[t][i+seq_len])
    return np.array(X), {t: np.array(y[t]).reshape(-1, 1) for t in target_cols}

X_train, y_train = make_sequences(train_df, seq_len)
X_val, y_val = make_sequences(val_df, seq_len)
X_test, y_test = make_sequences(test_df, seq_len)
print(f"X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}")

In [ ]:
factory = ModelFactory(config)
model = factory.build(input_dim=X_train.shape[2])
model.summary()

In [ ]:
trainer = Trainer(config)
model = trainer.fit_keras(model, X_train, y_train, X_val, y_val)

In [ ]:
evaluator = TimeSeriesEvaluator(config)
y_pred = {}
for t in target_cols:
    preds = model.predict(X_test, verbose=0)
    idx = list(target_cols).index(t)
    y_pred[t] = preds[idx].flatten()
y_true = {t: y_test[t].flatten() for t in target_cols}

results = evaluator.evaluate_all_targets(y_true, y_pred)
evaluator.print_report(results)

In [ ]:
for t in target_cols:
    evaluator.plot_predictions(y_true[t], y_pred[t], t)